<a href="https://colab.research.google.com/github/monasolgi/Computer_vision/blob/main/tutorial5_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Task 2– Batch Normalization in Python**

# Task 2 — Manual Implementation of Batch Normalization

In this task, we manually implement a Batch Normalization layer for a convolutional neural network using TensorFlow/Keras, without using the built-in `BatchNormalization` layer.

The goal is to understand:
- how Batch Normalization works internally,
- how normalization stabilizes deep neural network training,
- and why it improves convergence for very deep networks.

The network used in this exercise contains several convolution blocks with many convolution layers (`K` layers per block). For large values of `K` (e.g. `K=20`), training without normalization becomes unstable and may stall.


In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import keras
from keras import ops

import sys
import numpy as np
import matplotlib.pyplot as plt


class MyBatchNormalization(keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.epsilon = 1e-5

    def build(self, input_tensor):
      self.alpha = self.add_weight(
            shape=(input_tensor[-1],),
            initializer="ones",
            trainable=True,
        )
      self.beta = self.add_weight(shape=(input_tensor[-1],),
                                 initializer="zeros", trainable=True)

    def call(self,input_tensor):

      mean=tf.math.reduce_mean(
        input_tensor,
        axis=[0,1,2],
        keepdims=True,
        name=None
        )
      std=tf.math.reduce_std(
        input_tensor,
        axis=[0,1,2],
        keepdims=True,
        name=None
        )
      input_tensor = (input_tensor - mean) / (std+self.epsilon)


      return input_tensor* self.alpha + self.beta

def define_model():
    inputs = keras.Input(shape=(28,28,1))

    K = 3 # number of convolution layers per block
    L = 3  # number of blocks
    x = inputs
    for i in range(0,L):
        for j in range(0,K):
            # Add call to custom layer here: x = MyBatchNormalization()(x)
            x = MyBatchNormalization()(x)
            x = layers.Conv2D(32, 3, activation="relu",padding="same")(x)

        x = layers.MaxPooling2D(3)(x)
    x = layers.GlobalMaxPooling2D()(x)
    outputs = layers.Dense(10,activation='softmax')(x)

    model = keras.Model(inputs,outputs)
    model.summary() # show model overview
    return model


In [ ]:
# Load and preprocess training data (Fashion-MNIST)
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()
print(train_images.shape,test_images.shape)
#train_images = np.expand_dims(train_images, -1)
#test_images = np.expand_dims(test_images, -1)
train_images = train_images / 255.0
test_images = test_images / 255.0
train_labels = tf.keras.utils.to_categorical(train_labels)
test_labels = tf.keras.utils.to_categorical(test_labels)

# Define and train model
model = define_model()
model.compile(loss=keras.losses.CategoricalCrossentropy(),optimizer=keras.optimizers.Adam(),metrics=["accuracy"])
model.fit(train_images,train_labels, batch_size=64, epochs=100)


(60000, 28, 28) (10000, 28, 28)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_60       │ (None, 28, 28, 1)      │             2 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_62 (Conv2D)              │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_61       │ (None, 28, 28, 32)     │            64 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_63 (Conv2D)              │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_62       │ (None, 28, 28, 32)     │            64 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_64 (Conv2D)              │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 9, 9, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_63       │ (None, 9, 9, 32)       │            64 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_65 (Conv2D)              │ (None, 9, 9, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_64       │ (None, 9, 9, 32)       │            64 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_66 (Conv2D)              │ (None, 9, 9, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_65       │ (None, 9, 9, 32)       │            64 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_67 (Conv2D)              │ (None, 9, 9, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 3, 3, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_66       │ (None, 3, 3, 32)       │            64 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_68 (Conv2D)              │ (None, 3, 3, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_67       │ (None, 3, 3, 32)       │            64 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_69 (Conv2D)              │ (None, 3, 3, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_batch_normalization_68       │ (None, 3, 3, 32)       │            64 │
│ (MyBatchNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 75,148 (293.55 KB)

 Trainable params: 75,148 (293.55 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 191s 197ms/step - accuracy: 0.8419 - loss: 0.4386
Epoch 2/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 204s 200ms/step - accuracy: 0.9003 - loss: 0.2728
Epoch 3/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 182s 194ms/step - accuracy: 0.9142 - loss: 0.2330
Epoch 4/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 201s 192ms/step - accuracy: 0.9236 - loss: 0.2096
Epoch 5/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 202s 192ms/step - accuracy: 0.9291 - loss: 0.1911
Epoch 6/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 179s 191ms/step - accuracy: 0.9353 - loss: 0.1770
Epoch 7/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 178s 189ms/step - accuracy: 0.9387 - loss: 0.1632
Epoch 8/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 204s 191ms/step - accuracy: 0.9457 - loss: 0.1489
Epoch 9/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 203s 192ms/step - accuracy: 0.9474 - loss: 0.1405
Epoch 10/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 200s 190ms/step - accuracy: 0.9533 - loss: 0.1282
Epoch 11/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 178s 190ms/step - accuracy: 0.9557 - loss: 0.12